In [1]:
import numpy as np
import pandas as pd
import os

In [ ]:
"""You are a deterministic information extraction engine.
Your job is to convert a natural language Job Description (JD) into a STRICT, MACHINE-READY JSON STRUCTURE.

This JSON is designed for:
- candidate ranking systems
- filtering pipelines
- numerical scoring models
- SQL / vector hybrid databases
- ML-based matching engines

IMPORTANT:
- NO natural language output
- NO explanations
- NO extra keys
- NO missing fields
- NO free-text ambiguity except where explicitly allowed
- Every value must be structured for direct computation

------------------------------------------------------------
GLOBAL RULES
------------------------------------------------------------
1. Output MUST be valid JSON.
2. All keys must exist exactly as specified.
3. All categorical values must be normalized (UPPER_SNAKE_CASE).
4. All numeric values must be machine-computable (ints, floats).
5. No prose inside structured fields except "reason" fields.
6. If information is not present in JD:
   - use null for scalar fields
   - use [] for lists
   - use {} for dicts

------------------------------------------------------------
FINAL OUTPUT SCHEMA (STRICT)
------------------------------------------------------------
Return a single JSON object with this exact structure:
{
  "experience",
  "location",
  "company",
  "roles",
  "education",
  "skills",
  "communication",
  "work_style",
  "experience_requirements",
  "disqualifiers"
}

------------------------------------------------------------
1. EXPERIENCE (DICT)
------------------------------------------------------------
Structure:
{
  "min_years": int,
  "max_years": int,
  "ideal_years": {
    "min": int,
    "max": int
  }
}

Field meaning:
- min_years: minimum required experience from JD
- max_years: maximum allowed experience
- ideal_years: strongest hiring band inferred from JD

Data rules:
- Always integers
- Never strings
- Never null unless completely missing

------------------------------------------------------------
2. LOCATION (DICT)
------------------------------------------------------------
Structure:
{
  "domestic": {
    "preferred_cities": list[str],
    "secondary_cities": list[str],
    "relocation_allowed": bool,
    "preferred_relocation_targets": list[str]
  },
  "international": {
    "relocation_allowed": bool,
    "preferred_regions": list[str]
  }
}

Rules:
- Cities must be normalized strings
- relocation_allowed must be boolean
- international relocation inferred only if explicitly stated

------------------------------------------------------------
3. COMPANY (DICT)
------------------------------------------------------------
Structure:
{
  "size": {
    "most_preferred": {
      "min_employees": int,
      "max_employees": int,
      "weight": float
    },
    "good_to_have": {
      "min_employees": int,
      "max_employees": int,
      "weight": float
    }
  },

  "industry_constraints": {
    "excluded_industries": {
      "<INDUSTRY_NAME>": {
        "rule_type": str,
        "only_in_this_industry": bool,
        "allowed_if_part_of_other_industries": bool,
        "examples": list[str]
      }
    },

    "preferred_industries": list[str]
  }
}

Rules:
- company size MUST be numeric ranges only
- weight:
  - most_preferred = 1.0
  - good_to_have = 0.5–0.8
- industries MUST be normalized taxonomy labels
- excluded_industries keys MUST be industry categories, not companies

------------------------------------------------------------
4. ROLES (DICT)
------------------------------------------------------------
Structure:
{
  "preferred_titles": list[str]
}

Rules:
- Titles must be normalized uppercase snake case:
  e.g., ML_ENGINEER, SEARCH_ENGINEER
- Expand synonyms aggressively but only if implied

------------------------------------------------------------
5. EDUCATION (DICT)
------------------------------------------------------------
Structure:
{
  "degrees": {
    "bachelors": {
      "required": bool,
      "preferred": bool
    },
    "masters": {
      "required": bool,
      "preferred": bool
    },
    "phd": {
      "required": bool,
      "preferred": bool
    }
  },

  "fields_of_study": {
    "bachelors": list[str],
    "masters": list[str],
    "phd": list[str]
  },

  "grade_requirement": {
    "required": bool,
    "min_gpa": float | null,
    "min_percentage": float | null
  },

  "institution_tier": {
    "explicit_required": bool,
    "preferred_tiers": list[str]
  }
}

Rules:
- boolean flags must reflect explicit JD statements
- GPA/percentage must be numeric only
- if absent → null

------------------------------------------------------------
6. SKILLS (DICT)
------------------------------------------------------------
Structure:
{
  "core_required": [
    {
      "skill_id": str,
      "min_months": int,
      "proficiency": int,
      "mandatory": bool
    }
  ],

  "secondary_skills": [
    {
      "skill_id": str,
      "min_months": int,
      "proficiency": int,
      "mandatory": bool
    }
  ]
}

Rules:
- skill_id MUST be normalized taxonomy:
  EMBEDDINGS, VECTOR_DB, RANKING_SYSTEMS, PYTHON, etc.
- proficiency scale:
  1 = basic
  2 = working
  3 = intermediate
  4 = advanced
  5 = expert production
- min_months must be integer
- mandatory strictly from JD language:
  "must" → true

------------------------------------------------------------
7. COMMUNICATION (DICT)
------------------------------------------------------------
Structure:
{
  "languages": [
    {
      "language_id": str,
      "required_proficiency": int,
      "mandatory": bool
    }
  ]
}

Rules:
- proficiency scale same as skills (1–5)
- default language: ENGLISH if implied

------------------------------------------------------------
8. WORK_STYLE (DICT)
------------------------------------------------------------
Structure:
{
  "communication_model": str,
  "decision_style": str,
  "engineering_style": str,
  "product_philosophy": str,
  "ownership_expectation": str
}

Rules:
- ALL values MUST be enum-like constants:
  e.g.,
  ASYNC_WRITING_HEAVY
  FAST_ITERATION
  USER_FEEDBACK_DRIVEN
  FULL_STACK_ML_SYSTEMS

------------------------------------------------------------
9. EXPERIENCE_REQUIREMENTS (DICT)
------------------------------------------------------------
Structure:
{
  "required_domains": [
    {
      "domain_id": str,
      "min_years": int
    }
  ]
}

Rules:
- domain_id must be normalized:
  SEARCH_SYSTEMS, RANKING_SYSTEMS, RETRIEVAL_SYSTEMS, ML_PRODUCTION_SYSTEMS

------------------------------------------------------------
10. DISQUALIFIERS (DICT)
------------------------------------------------------------
Structure:
{
  "career_paths": [
    {
      "type": str,
      "severity": str,
      "examples": list[str]
    }
  ],

  "skill_gaps": list[str],

  "behavioral": list[str],

  "tooling_only_profiles": list[str],

  "career_patterns": list[str]
}

Rules:
- type MUST be categorical machine label
- severity MUST be:
  HIGH | MEDIUM | LOW
- all strings must be normalized uppercase snake case
- no free text sentences allowed

------------------------------------------------------------
CRITICAL DESIGN GOAL
------------------------------------------------------------
This schema is intentionally:
- fully numeric where possible
- fully categorical where possible
- ML-ranking ready
- SQL filterable
- vector searchable
- free-text minimized

------------------------------------------------------------
FINAL INSTRUCTION
------------------------------------------------------------
Given any JD:
1. Extract information
2. Normalize into schema
3. Ensure strict typing
4. Output ONLY JSON
5. Never include explanations"""

In [13]:
from json import loads, dump

job_requirements = loads("""{
"experience": {
"min_years": 5,
"max_years": 9,
"ideal_years": {
"min": 6,
"max": 8
}
},
"location": {
"domestic": {
"preferred_cities": [
"PUNE",
"NOIDA"
],
"secondary_cities": [
"HYDERABAD",
"MUMBAI",
"DELHI_NCR"
],
"relocation_allowed": true,
"preferred_relocation_targets": [
"PUNE",
"NOIDA"
]
},
"international": {
"relocation_allowed": false,
"preferred_regions": []
}
},
"company": {
"size": {
"most_preferred": {
"min_employees": 10,
"max_employees": 50,
"weight": 1.0
},
"good_to_have": {
"min_employees": 50,
"max_employees": 200,
"weight": 0.6
}
},
"industry_constraints": {
"excluded_industries": {},
"preferred_industries": [
"AI_NATIVE_TALENT_INTELLIGENCE",
"RECRUITMENT_TECH",
"SEARCH_INFRASTRUCTURE",
"RECOMMENDATION_SYSTEMS",
"HR_TECH",
"APPLIED_ML"
]
}
},
"roles": {
"preferred_titles": [
"SENIOR_AI_ENGINEER",
"ML_ENGINEER",
"SEARCH_ENGINEER",
"RANKING_ENGINEER",
"RECOMMENDER_SYSTEMS_ENGINEER",
"APPLIED_SCIENTIST"
]
},
"education": {
"degrees": {
"bachelors": {
"required": false,
"preferred": false
},
"masters": {
"required": false,
"preferred": false
},
"phd": {
"required": false,
"preferred": false
}
},
"fields_of_study": {
"bachelors": [],
"masters": [],
"phd": []
},
"grade_requirement": {
"required": false,
"min_gpa": null,
"min_percentage": null
},
"institution_tier": {
"explicit_required": false,
"preferred_tiers": []
}
},
"skills": {
"core_required": [
{
"skill_id": "EMBEDDINGS",
"min_months": 24,
"proficiency": 5,
"mandatory": true
},
{
"skill_id": "RETRIEVAL_SYSTEMS",
"min_months": 24,
"proficiency": 5,
"mandatory": true
},
{
"skill_id": "VECTOR_DB",
"min_months": 24,
"proficiency": 4,
"mandatory": true
},
{
"skill_id": "HYBRID_SEARCH",
"min_months": 24,
"proficiency": 4,
"mandatory": true
},
{
"skill_id": "PYTHON",
"min_months": 36,
"proficiency": 5,
"mandatory": true
},
{
"skill_id": "RANKING_SYSTEMS_EVALUATION",
"min_months": 24,
"proficiency": 5,
"mandatory": true
},
{
"skill_id": "EVALUATION_FRAMEWORKS",
"min_months": 24,
"proficiency": 5,
"mandatory": true
},
{
"skill_id": "AB_TESTING",
"min_months": 18,
"proficiency": 4,
"mandatory": true
}
],
"secondary_skills": [
{
"skill_id": "LLM_FINE_TUNING",
"min_months": 12,
"proficiency": 3,
"mandatory": false
},
{
"skill_id": "LEARNING_TO_RANK",
"min_months": 12,
"proficiency": 4,
"mandatory": false
},
{
"skill_id": "HR_TECH",
"min_months": 6,
"proficiency": 3,
"mandatory": false
},
{
"skill_id": "DISTRIBUTED_SYSTEMS",
"min_months": 12,
"proficiency": 3,
"mandatory": false
},
{
"skill_id": "OPEN_SOURCE_CONTRIBUTION",
"min_months": 6,
"proficiency": 2,
"mandatory": false
}
]
},
"communication": {
"languages": [
{
"language_id": "ENGLISH",
"required_proficiency": 4,
"mandatory": true
}
]
},
"work_style": {
"communication_model": "ASYNC_WRITING_HEAVY",
"decision_style": "FAST_ITERATION",
"engineering_style": "FULL_STACK_ML_SYSTEMS",
"product_philosophy": "USER_FEEDBACK_DRIVEN",
"ownership_expectation": "HIGH_OWNERSHIP"
},
"experience_requirements": {
"required_domains": [
{
"domain_id": "RETRIEVAL_SYSTEMS",
"min_years": 3
},
{
"domain_id": "RANKING_SYSTEMS",
"min_years": 3
},
{
"domain_id": "SEARCH_SYSTEMS",
"min_years": 2
},
{
"domain_id": "ML_PRODUCTION_SYSTEMS",
"min_years": 3
}
]
},
"disqualifiers": {
"career_paths": [
{
"type": "PURE_RESEARCH",
"severity": "HIGH",
"examples": [
"ACADEMIC_LABS_ONLY",
"NO_PRODUCTION_DEPLOYMENT"
]
},
{
"type": "TOOLING_ONLY_AI",
"severity": "HIGH",
"examples": [
"LANGCHAIN_ONLY_PROJECTS",
"RECENT_LLM_WRAPPER_ONLY_EXPERIENCE"
]
},
{
"type": "CONSULTING_ONLY",
"severity": "HIGH",
"examples": [
"TCS_INFOSYS_WIPRO_ONLY_BACKGROUND",
"NO_PRODUCT_COMPANY_EXPERIENCE"
]
},
{
"type": "CAREER_HOPPING_TITLE_OPTIMIZATION",
"severity": "MEDIUM",
"examples": [
"JOB_SWITCH_EVERY_18_MONTHS",
"TITLE_ASCENSION_OPTIMIZATION"
]
},
{
"type": "DOMAIN_MISMATCH",
"severity": "HIGH",
"examples": [
"COMPUTER_VISION_ONLY",
"ROBOTICS_ONLY",
"SPEECH_ONLY_WITH_NO_NLP_IR"
]
}
],
"skill_gaps": [
"NO_PRODUCTION_RANKING_SYSTEMS",
"NO_VECTOR_DATABASE_EXPERIENCE",
"NO_EVALUATION_FRAMEWORKS",
"NO_RETRIEVAL_SYSTEMS_EXPERIENCE"
],
"behavioral": [
"LOW_OWNERSHIP",
"SLOW_ITERATION",
"AVOID_WRITING",
"AVERSION_TO_AMBIGUITY"
],
"tooling_only_profiles": [
"LANGCHAIN_TUTORIAL_BUILDERS",
"DEMO_ONLY_ML_PROJECTS",
"FRAMEWORK_CHASERS"
],
"career_patterns": [
"FREQUENT_JOB_SWITCHING_18_MONTHS",
"TITLE_OPTIMIZATION_PATH",
"CONSULTING_ONLY_CAREER_PATH"
]
}
}

""")

In [14]:
with open("job_requirements.json", "w") as f:
    dump(job_requirements, f, indent=4)

In [2]:
data_dir = "data/raw/"
data_files = sorted([data_dir + file for file in os.listdir(data_dir) if file.endswith(".csv")])
data_files

['data/raw/candidate_career_history.csv',
 'data/raw/candidate_certifications.csv',
 'data/raw/candidate_education.csv',
 'data/raw/candidate_education_enriched.csv',
 'data/raw/candidate_languages.csv',
 'data/raw/candidate_org.csv',
 'data/raw/candidate_profile.csv',
 'data/raw/candidate_redrob_signals.csv',
 'data/raw/candidate_skills.csv',
 'data/raw/certification_issuer_features.csv']

In [3]:
def prepare_df_summary(df):
    return pd.DataFrame({
        "dtype": df.dtypes,
        "null_count": df.isna().sum(),
        "unique_count": df.nunique()
    })

In [25]:
profile_df = pd.read_csv("data/raw/candidate_profile.csv")
profile_df.to_csv("data/cleaned/candidate_profile.csv", index=False)
# len(profile_df), prepare_df_summary(profile_df)
profile_df.head()

,candidate_id,name,headline,summary,location,country,years_of_exp,current_title,current_company,current_company_size,current_industry
0,CAND_0000001,Ira Vora,"Backend Engineer | SQL, Spark, Cloud",Software / data professional with 6.9 years of...,Toronto,Canada,6.9,Backend Engineer,Mindtree,10001+,IT Services
1,CAND_0000002,Saanvi Sethi,Operations Manager | 12.5+ yrs experience,Professional with 12.5+ years of experience. M...,"Chennai, Tamil Nadu",India,12.5,Operations Manager,Wipro,10001+,IT Services
2,CAND_0000003,Yash Agarwal,Customer Support | 1.1+ yrs experience,Professional with 1.1+ years of experience. I'...,Austin,USA,1.1,Customer Support,TCS,10001+,IT Services
3,CAND_0000004,Anil Bose,Marketing Manager | Driving business outcomes,Professional with 3.8+ years of experience. My...,Sydney,Australia,3.8,Marketing Manager,Dunder Mifflin,201-500,Paper Products
4,CAND_0000005,Aisha Sethi,Accountant | Helping teams scale,Professional with 11.0+ years of experience. I...,"Gurgaon, Haryana",India,11.0,Accountant,Stark Industries,1001-5000,Manufacturing


In [23]:
redrob_signals_df = pd.read_csv("data/raw/candidate_redrob_signals.csv")
redrob_signals_df.to_csv("data/cleaned/candidate_redrob_signals.csv", index=False)
len(redrob_signals_df), prepare_df_summary(redrob_signals_df)

(100000,
                               dtype  null_count  unique_count
 candidate_id                 object           0        100000
 profile_completeness_score  float64           0           729
 signup_date                  object           0          1496
 last_active_date             object           0           241
 open_to_work_flag              bool           0             2
 profile_views_received_30d    int64           0           261
 applications_submitted_30d    int64           0            25
 recruiter_response_rate     float64           0            94
 avg_response_time_hours     float64           0          2779
 skill_assessment_scores      object           0         18685
 connection_count              int64           0          1191
 endorsements_received         int64           0           172
 notice_period_days            int64           0             8
 expected_min_salary_lpa     float64           0           397
 expected_max_salary_lpa     float64          

In [22]:
career_df = pd.read_csv("data/raw/candidate_career_history.csv")
career_df = career_df.fillna(value="")
career_df.to_csv("data/cleaned/candidate_career_history.csv", index=False)
len(career_df), prepare_df_summary(career_df)

(300171,
                   dtype  null_count  unique_count
 candidate_id     object           0        100000
 company          object           0            63
 title            object           0            48
 start_date       object           0          1048
 end_date         object           0          1075
 duration_months   int64           0            73
 is_current         bool           0             2
 industry         object           0            24
 company_size     object           0             7
 description      object           0            44)

In [21]:
education_df = pd.read_csv("data/raw/candidate_education.csv")
ed_inst_details_df = pd.read_csv("data/raw/candidate_education_enriched.csv")
education_df = education_df.merge(ed_inst_details_df, how="left", on=["candidate_id", "institution", "tier"])
education_df.to_csv("data/cleaned/candidate_education.csv", index=False)
len(education_df), prepare_df_summary(education_df)

(145870,
                        dtype  null_count  unique_count
 candidate_id          object           0        100000
 institution           object           0            44
 degree                object           0             8
 field_of_study        object           0            16
 start_year             int64           0            20
 end_year               int64           0            18
 grade                 object           0           329
 tier                  object           0             4
 national_ranking       int64           0            35
 global_ranking         int64           0            25
 industry_connections   int64           0             6
 quality_of_course      int64           0             3)

In [20]:
cert_df = pd.read_csv("data/raw/candidate_certifications.csv")
cert_issuer_details_df = pd.read_csv("data/raw/certification_issuer_features.csv")
cert_issuer_details_df = cert_issuer_details_df.drop_duplicates(ignore_index=True)
cert_enriched_df = cert_df.merge(cert_issuer_details_df, how="left", on=["candidate_id", "issuer"])
cert_enriched_df.to_csv("data/cleaned/candidate_certifications.csv", index=False)
len(cert_enriched_df), prepare_df_summary(cert_enriched_df)

(37484,
                                   dtype  null_count  unique_count
 candidate_id                     object           0         24981
 name                             object           0             8
 issuer                           object           0             6
 year                              int64           0             9
 issuer_base_location             object           0             1
 issue_location                   object           0             1
 issuer_establishment_age          int64           0             5
 issuer_recognition_in_community   int64           0             6)

In [18]:
skills_df = pd.read_csv("data/raw/candidate_skills.csv")
skills_df["endorsements"] = (skills_df["endorsements"] - skills_df["endorsements"].min()) / (skills_df["endorsements"].max() - skills_df["endorsements"].min())
skills_df.to_csv("data/cleaned/candidate_skills.csv", index=False)
len(skills_df), prepare_df_summary(skills_df)


(960302,
                    dtype  null_count  unique_count
 candidate_id      object           0        100000
 name              object           0           133
 proficiency       object           0             4
 endorsements     float64           0            61
 duration_months    int64           0            96)

In [100]:
skill_proficiency_weights = {prof: weight for prof, weight in zip(sorted(skills_df["proficiency"].unique()), [1., 0.25, 0.75, 0.5])}

In [16]:
languages_df = pd.read_csv("data/raw/candidate_languages.csv")
languages_df.to_csv("data/cleaned/candidate_languages.csv", index=False)
len(languages_df), prepare_df_summary(languages_df)

(200000,
                dtype  null_count  unique_count
 candidate_id  object           0        100000
 language      object           0             2
 proficiency   object           0             3)

In [89]:
language_weights = {lang: weight for lang, weight in zip(sorted(languages_df["language"].unique()), [0.5, 0.5])}
proficiency_weights = {prof: weight for prof, weight in zip(sorted(languages_df["proficiency"].unique()), [0.3, 0.4, 0.5])}

language_score = lambda language, proficiency: language_weights[language] + proficiency_weights[proficiency]